# LLM Data Preprocessing Pipeline

> End-to-end walkthrough: raw text → input embeddings ready for Transformer training.

---

## Pipeline Overview

```
Raw Text
   │  Stage 1 ─ Tokenization
   ▼
Tokenized Text  (list of string tokens)
   │  Stage 2 ─ Vocabulary + Token IDs
   ▼
Token IDs  (integers)
   │  Stage 3 ─ Token Embeddings  +  DataLoader
   ▼
Token Embedding Vectors  (B × S × D  float tensor)
   │  Stage 4 ─ Positional Embeddings
   ▼
Positional Embedding Vectors  (S × D  float tensor)
   │  Stage 5 ─ Combine
   ▼
Input Embeddings  (B × S × D)  ←  ready for the Transformer
```

`B` = batch size · `S` = sequence length · `D` = embedding dimension

---

## Table of Contents
1. [Stage 1 — Raw Text to Tokenized Text](#stage-1)
2. [Stage 2 — Tokenized Text to Token IDs](#stage-2)
3. [Stage 3 — Token IDs to Token Embeddings + DataLoader](#stage-3)
4. [Stage 4 — Token Embeddings to Positional Embeddings](#stage-4)
5. [Stage 5 — Final Input Embeddings](#stage-5)


## Setup

Install and import all required libraries before running any stage.

In [ ]:
# Uncomment the line below if tiktoken is not yet installed
# !pip install tiktoken

import re
import importlib
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

print(f"PyTorch  : {torch.__version__}")
print(f"tiktoken : {importlib.metadata.version('tiktoken')}")


In [ ]:
# Load the sample corpus — Edith Wharton's short story "The Verdict"
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Corpus length : {len(raw_text):,} characters")
print(f"Preview       : {raw_text[:120]}")


---
<a id="stage-1"></a>
## Stage 1 — Raw Text to Tokenized Text

**Goal:** Split the continuous character stream into a list of meaningful string tokens.

A regular expression pattern handles whitespace and punctuation simultaneously,
producing one token per word and one per punctuation mark.

> **Design note:** Whitespace tokens are discarded after splitting. Retaining them
> increases sequence length without adding semantic value for most LLM tasks.


In [ ]:
# Pattern splits on whitespace and common punctuation marks
SPLIT_PATTERN = r'([,.:;?_!"()\']|--|\s)'

tokenized = re.split(SPLIT_PATTERN, raw_text)
tokenized = [tok.strip() for tok in tokenized if tok.strip()]

print(f"Total tokens : {len(tokenized):,}")
print(f"First 20     : {tokenized[:20]}")


---
<a id="stage-2"></a>
## Stage 2 — Tokenized Text to Token IDs

**Goal:** Convert string tokens into integer IDs via a vocabulary lookup table.

### 2a. Vocabulary Construction

All unique tokens are sorted alphabetically and assigned sequential integers.
Two special tokens are always appended:

| Token | Purpose |
|---|---|
| `<\|unk\|>` | Fallback for any word absent from the vocabulary |
| `<\|endoftext\|>` | Boundary marker between independent documents |

### 2b. Tokenizer Implementations

| Class | OOV handling | When to use |
|---|---|---|
| `TokenizerV1` | Raises `KeyError` | Closed-corpus / known vocabulary |
| `TokenizerV2` | Maps to `<\|unk\|>` | Open-domain / production |


In [ ]:
unique_tokens = sorted(set(tokenized))
unique_tokens += ["<|endoftext|>", "<|unk|>"]   # special tokens appended last

vocab      = {token: idx for idx, token in enumerate(unique_tokens)}
vocab_size = len(vocab)

print(f"Vocabulary size : {vocab_size:,} tokens")
print(f"Last 6 entries  : {list(vocab.items())[-6:]}")


In [ ]:
class TokenizerV1:
    """
    Vocab-based tokenizer with no OOV tolerance.
    Raises KeyError when an unseen token is encountered during encode().
    Best for closed-corpus scenarios where the full vocabulary is guaranteed.
    """

    def __init__(self, vocab: dict):
        self.str_to_int = vocab
        self.int_to_str = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text: str) -> list:
        tokens = re.split(SPLIT_PATTERN, text)
        tokens = [t.strip() for t in tokens if t.strip()]
        return [self.str_to_int[t] for t in tokens]   # KeyError on OOV

    def decode(self, ids: list) -> str:
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)


# Quick demo
tok_v1  = TokenizerV1(vocab)
passage = '"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'
ids_v1  = tok_v1.encode(passage)
print("Encoded :", ids_v1)
print("Decoded :", tok_v1.decode(ids_v1))


In [ ]:
class TokenizerV2:
    """
    OOV-tolerant tokenizer.
    Replaces any unseen word with <|unk|> instead of raising an error.
    Also handles <|endoftext|> for separating independent documents.
    """

    def __init__(self, vocab: dict):
        self.str_to_int = vocab
        self.int_to_str = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text: str) -> list:
        tokens = re.split(SPLIT_PATTERN, text)
        tokens = [t.strip() for t in tokens if t.strip()]
        tokens = [t if t in self.str_to_int else "<|unk|>" for t in tokens]
        return [self.str_to_int[t] for t in tokens]

    def decode(self, ids: list) -> str:
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)


# Demo: two OOV words joined by <|endoftext|>
tok_v2 = TokenizerV2(vocab)
doc_a  = "Hello, do you like tea?"                # "Hello" → <|unk|>
doc_b  = "In the sunlit terraces of the palace."  # "palace" → <|unk|>
joined = " <|endoftext|> ".join([doc_a, doc_b])

print("Input  :", joined)
print("IDs    :", tok_v2.encode(joined))
print("Decoded:", tok_v2.decode(tok_v2.encode(joined)))


### 2c. Byte Pair Encoding (BPE) via tiktoken

Production GPT models use **Byte Pair Encoding** instead of regex splitting.
BPE iteratively merges the most frequent adjacent byte pairs to build a sub-word
vocabulary. Any string can be represented without ever needing `<|unk|>`.

We load OpenAI's pre-trained GPT-2 BPE encoding via `tiktoken`.


In [ ]:
bpe_tokenizer = tiktoken.get_encoding("gpt2")
print(f"BPE vocabulary size : {bpe_tokenizer.n_vocab:,}")

sample = (
    "Hello, do you like tea? <|endoftext|> "
    "In the sunlit terraces of someunknownPlace."
)
bpe_ids = bpe_tokenizer.encode(sample, allowed_special={"<|endoftext|>"})
print(f"Token IDs : {bpe_ids}")
print(f"Decoded   : {bpe_tokenizer.decode(bpe_ids)}")

# BPE splits unknown strings into sub-word pieces — never produces <|unk|>
novel_ids = bpe_tokenizer.encode("Akwirw ier")
print(f"\n'Akwirw ier' → IDs    : {novel_ids}")
print(f"             → pieces : {[bpe_tokenizer.decode([i]) for i in novel_ids]}")


---
<a id="stage-3"></a>
## Stage 3 — Token IDs to Token Embeddings + DataLoader

**Goal:** Map integer token IDs to dense float vectors, and build the batched
training pipeline using a sliding-window DataLoader.

### Sliding-Window Input–Target Pairs

LLMs train on a **next-token prediction** objective. A window of width `max_length`
steps through the full token sequence, creating `(input, target)` pairs where the
target is the input shifted right by one position.

```
tokens  : [t0, t1, t2, t3, t4, t5, ...]
input x : [t0, t1, t2, t3]
target y: [t1, t2, t3, t4]   ← shifted by 1
```

The `stride` controls window overlap:
- `stride = 1` → maximum overlap, maximum data reuse
- `stride = max_length` → no overlap, cleanest batches

### Token Embedding Layer

`nn.Embedding(vocab_size, embed_dim)` is a learnable lookup table.
Row `i` of its weight matrix is returned when indexed with token ID `i`.
These weights are randomly initialised and refined during training.


In [ ]:
class GPTDatasetV1(Dataset):
    """
    Converts a raw text corpus into overlapping (input, target) tensor pairs
    using a sliding context window — the standard setup for next-token prediction.

    Parameters
    ----------
    txt        : full corpus as a single string
    tokenizer  : tiktoken-compatible encoder
    max_length : context window width (sequence length)
    stride     : tokens to advance between consecutive windows
    """

    def __init__(self, txt, tokenizer, max_length, stride):
        all_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.input_ids  = []
        self.target_ids = []

        for start in range(0, len(all_ids) - max_length, stride):
            self.input_ids.append(
                torch.tensor(all_ids[start : start + max_length])
            )
            self.target_ids.append(
                torch.tensor(all_ids[start + 1 : start + max_length + 1])
            )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [ ]:
def create_dataloader_v1(
    txt,
    batch_size : int  = 4,
    max_length : int  = 256,
    stride     : int  = 128,
    shuffle    : bool = True,
    drop_last  : bool = True,
    num_workers: int  = 0,
) -> DataLoader:
    """
    Wraps GPTDatasetV1 in a PyTorch DataLoader.

    Parameters
    ----------
    batch_size   : sequences per mini-batch
    max_length   : context window size
    stride       : step between windows (< max_length → overlapping windows)
    shuffle      : randomise order each epoch — recommended for training
    drop_last    : discard final incomplete batch to avoid shape mismatches
    num_workers  : worker processes for parallel data loading
    """
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset   = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = shuffle,
        drop_last   = drop_last,
        num_workers = num_workers,
    )


In [ ]:
# ── Verify DataLoader output ──────────────────────────────────────────────────
# stride=1 → each consecutive batch shifts by exactly one token
dl_debug = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
it      = iter(dl_debug)
batch_a = next(it)
batch_b = next(it)
print("Batch A  inputs:", batch_a[0], " targets:", batch_a[1])
print("Batch B  inputs:", batch_b[0], " targets:", batch_b[1])
print("(Batch B's input starts where Batch A's had its 2nd token — stride=1 overlap)")


In [ ]:
# ── Token Embedding Layer ─────────────────────────────────────────────────────
VOCAB_SIZE = 50_257   # GPT-2 BPE vocabulary
EMBED_DIM  = 256      # embedding dimension  (GPT-3 uses 12,288; 256 for demo)
MAX_LENGTH = 4        # context window / sequence length

torch.manual_seed(42)
token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, EMBED_DIM)

# Load one batch (non-overlapping windows, batch_size=8)
dl_train = create_dataloader_v1(
    raw_text,
    batch_size = 8,
    max_length = MAX_LENGTH,
    stride     = MAX_LENGTH,   # stride = max_length → no overlap
    shuffle    = False,
)
inputs, targets = next(iter(dl_train))

print(f"Token ID tensor shape  : {inputs.shape}")            # (8, 4)

token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape : {token_embeddings.shape}")  # (8, 4, 256)


---
<a id="stage-4"></a>
## Stage 4 — Token Embeddings to Positional Embeddings

**Goal:** Produce a vector for each *position* in the sequence so the model can
distinguish token order.

### Why positional encoding?

Transformer self-attention is **permutation-invariant** — it treats its input
as a set, not an ordered sequence. Without position information, the model
cannot tell `[A, B, C]` apart from `[C, A, B]`.

### Absolute learned positional embeddings (GPT-2 style)

A second `nn.Embedding` of shape `(context_length, embed_dim)` maps each
position index `0, 1, ..., context_length−1` to its own learned vector.
These weights are updated during training alongside the token embeddings.


In [ ]:
pos_embedding_layer = torch.nn.Embedding(MAX_LENGTH, EMBED_DIM)

# Input: sequential position indices [0, 1, 2, ..., MAX_LENGTH-1]
position_indices = torch.arange(MAX_LENGTH)               # shape: (4,)
pos_embeddings   = pos_embedding_layer(position_indices)  # shape: (4, 256)

print(f"Position index shape       : {position_indices.shape}")
print(f"Positional embeddings shape: {pos_embeddings.shape}")
print("Each row encodes one position within the context window.")


---
<a id="stage-5"></a>
## Stage 5 — Final Input Embeddings

**Goal:** Fuse token identity and positional context into a single tensor —
the actual input fed into the first Transformer layer.

```
input_embeddings = token_embeddings + pos_embeddings
   (8, 4, 256)   +     (4, 256)     →  (8, 4, 256)   ← via broadcasting
```

PyTorch automatically broadcasts `pos_embeddings` of shape `(S, D)` across
all `B` samples in the batch. Each resulting 256-dim vector now encodes both
*what* the token is and *where* it appears in the sequence.


In [ ]:
# Broadcasting: (8, 4, 256) + (4, 256) → (8, 4, 256)
input_embeddings = token_embeddings + pos_embeddings

print(f"Token emb shape : {token_embeddings.shape}")   # (8, 4, 256)
print(f"Pos emb shape   : {pos_embeddings.shape}")     # (4, 256)  ← broadcast
print(f"Input emb shape : {input_embeddings.shape}")   # (8, 4, 256)
print("\ninput_embeddings is ready to be passed into the Transformer.")


---
## Pipeline Summary

| Stage | Input | Output | Key Tool |
|---|---|---|---|
| **1 — Tokenization** | Raw text string | `list[str]` tokens | `re.split` |
| **2 — Token IDs** | `list[str]` tokens | `list[int]` IDs | `vocab` dict / `tiktoken` BPE |
| **3 — Token Embeddings** | Batched IDs `(B, S)` | Float tensor `(B, S, D)` | `nn.Embedding(vocab_size, D)` |
| **4 — Positional Embeddings** | Position indices `(S,)` | Float tensor `(S, D)` | `nn.Embedding(max_length, D)` |
| **5 — Input Embeddings** | Token emb + Pos emb | Float tensor `(B, S, D)` | Elementwise `+` with broadcast |

---

## Key Hyperparameters

| Name | Demo value | Typical GPT-2 | Effect |
|---|---|---|---|
| `VOCAB_SIZE` | 50,257 | 50,257 | Rows in the token embedding table |
| `EMBED_DIM` | 256 | 768 | Width of every embedding vector |
| `MAX_LENGTH` | 4 | 1,024 | Tokens per forward pass (context window) |
| `stride` | 4 | 512 | Overlap between consecutive training windows |
| `batch_size` | 8 | 512 | Sequences processed per gradient update |
